# SpotCheck cross-check

An independent Python reimplementation of the TypeScript engine, written from the formulas in `SPOTCHECK_PLAN.md` rather than ported from the TS code. Each section draws seeded random inputs, computes the expected values here, and writes them to `engine/__tests__/fixtures/`. The Vitest suites then assert the TypeScript engine matches.

Run from the repo root's `notebooks/` folder with numpy and scipy installed. Re-running produces identical fixtures (fixed seeds).

In [1]:
import json
from datetime import date, timedelta
from pathlib import Path

import numpy as np

ROOT = Path.cwd().parent
DATA = ROOT / "data"
FIXTURES = ROOT / "engine" / "__tests__" / "fixtures"

def load_rows(rel):
    return json.loads((DATA / rel).read_text())["rows"]

def d(s):
    return date.fromisoformat(s)

usd_rows = load_rows("fx/SGD-USD.json")
usd_dates = [r[0] for r in usd_rows]
usd = np.array([r[2] for r in usd_rows])            # SGD per USD
sora_rows = load_rows("rates/SORA.json")
sora_dates = [r[0] for r in sora_rows]
sora = np.array([r[2] for r in sora_rows])          # decimal, by value date
dff_rows = load_rows("rates/USD-DFF.json")
dff_dates = [r[0] for r in dff_rows]
dff = np.array([r[1] for r in dff_rows])

def write_fixture(name, payload):
    (FIXTURES / name).write_text(json.dumps(payload, indent=2) + "\n")
    print(f"wrote {name}: {len(payload['cases'])} cases")

print(len(usd), "USD rows,", len(sora), "SORA rows,", len(dff), "DFF rows")

7116 USD rows, 5344 SORA rows, 7958 DFF rows


## 1. Compounded overnight rates

MAS method: $r = \left(\prod_i (1 + r_i n_i / B) - 1\right) \times 365 / N$, where every calendar day in $[\text{from}, \text{to})$ accrues at the latest rate published on or before it. $B = 365$ for SORA and $360$ for USD fed funds; the result is a simple ACT/365 rate.

This version walks the calendar day by day, counts how many days each published rate covers ($n_i$), and only then applies $(1 + r_i n_i / B)$, so it is structured differently from the TS engine. (An earlier draft compounded every calendar day separately, $(1 + r_i/B)^{n_i}$. The cross-check caught that: it differs from MAS by up to 0.03 bp.)

In [2]:
def compounded(dates, values, start, end, basis):
    idx = np.searchsorted(dates, start, side="right") - 1   # last date <= start
    assert idx >= 0
    days_per_rate = {}                                     # n_i for each rate index i
    day = d(start)
    while day < d(end):
        while idx + 1 < len(dates) and d(dates[idx + 1]) <= day:
            idx += 1
        days_per_rate[idx] = days_per_rate.get(idx, 0) + 1
        day += timedelta(days=1)
    growth = np.prod([1 + values[i] * n / basis for i, n in days_per_rate.items()])
    n_total = (d(end) - d(start)).days
    return (growth - 1) * 365 / n_total

rng = np.random.default_rng(20261017)
cases = []
for series, dates, values, basis in [("SORA", sora_dates, sora, 365), ("USD-DFF", dff_dates, dff, 360)]:
    for _ in range(10):
        start = d("2006-01-01") + timedelta(days=int(rng.integers(0, 7000)))
        length = int(rng.integers(7, 400))
        end = start + timedelta(days=length)
        cases.append({"series": series, "from": start.isoformat(), "to": end.isoformat(),
                      "basis": basis, "expected": compounded(dates, values, start.isoformat(), end.isoformat(), basis)})
write_fixture("compounding-crosscheck.json", {"source": "notebooks/crosscheck.ipynb", "cases": cases})

wrote compounding-crosscheck.json: 20 cases


## 2. Strategies

Each conversion costs units × rate × (1 + spread) at the first rate on or after its scheduled date. Tranche dates are spaced evenly in calendar days, start to deadline inclusive, rounded half up. The target strategy converts on the first day whose rate is at or below the target, else on the deadline.

In [3]:
def rate_on(day):
    i = int(np.searchsorted(usd_dates, day.isoformat(), side="left"))
    return usd[i]

def strategies(start, deadline, amount, k, spread, target):
    span = (deadline - start).days
    cost = lambda units, day: units * rate_on(day) * (1 + spread)
    lump = cost(amount, start)
    if k == 1:
        tr = lump
    else:
        days = [start + timedelta(days=int(np.floor(i * span / (k - 1) + 0.5))) for i in range(k)]
        tr = sum(cost(amount / k, day) for day in days)
    wait = cost(amount, deadline)
    tgt = wait
    i = int(np.searchsorted(usd_dates, start.isoformat(), side="left"))
    while i < len(usd_dates) and usd_dates[i] <= deadline.isoformat():
        if usd[i] <= target:
            tgt = amount * usd[i] * (1 + spread)
            break
        i += 1
    return {"lumpSum": lump, "tranches": tr, "wait": wait, "target": tgt}

rng = np.random.default_rng(1999)
cases = []
for _ in range(20):
    start = d("1999-02-01") + timedelta(days=int(rng.integers(0, 9000)))
    deadline = start + timedelta(days=int(rng.integers(30, 400)))
    amount = float(rng.integers(1, 50) * 1000)
    k = int(rng.integers(1, 13))
    spread = float(rng.choice([0.0, 0.0025, 0.005, 0.01]))
    target = float(rate_on(start) * (1 - rng.uniform(0, 0.05)))
    cases.append({"startDate": start.isoformat(), "deadline": deadline.isoformat(), "amount": amount,
                  "tranches": k, "spread": spread, "targetRate": target,
                  "expected": strategies(start, deadline, amount, k, spread, target)})
write_fixture("strategy-crosscheck.json", {"source": "notebooks/crosscheck.ipynb", "series": "SGD-USD", "cases": cases})

wrote strategy-crosscheck.json: 20 cases


## 3. Garman-Kohlhagen call and Greeks

$C = S e^{-r_f T} N(d_1) - K e^{-r_d T} N(d_2)$, $d_1 = \frac{\ln(S/K) + (r_d - r_f + \sigma^2/2)T}{\sigma\sqrt{T}}$, $d_2 = d_1 - \sigma\sqrt{T}$, with continuously compounded rates. Greeks use scipy's normal distribution:

- delta $= e^{-r_f T} N(d_1)$
- gamma $= e^{-r_f T} \varphi(d_1) / (S \sigma \sqrt{T})$
- vega $= S e^{-r_f T} \varphi(d_1) \sqrt{T}$ (per 1.00 of volatility)
- theta $= -\frac{S e^{-r_f T} \varphi(d_1) \sigma}{2\sqrt{T}} + r_f S e^{-r_f T} N(d_1) - r_d K e^{-r_d T} N(d_2)$ (per year)

In [4]:
from scipy.stats import norm

def gk(S, K, rd, rf, sigma, T):
    sq = np.sqrt(T)
    d1 = (np.log(S / K) + (rd - rf + sigma**2 / 2) * T) / (sigma * sq)
    d2 = d1 - sigma * sq
    df_f, df_d = np.exp(-rf * T), np.exp(-rd * T)
    call = S * df_f * norm.cdf(d1) - K * df_d * norm.cdf(d2)
    put = K * df_d * norm.cdf(-d2) - S * df_f * norm.cdf(-d1)
    return {
        "call": call, "put": put,
        "delta": df_f * norm.cdf(d1),
        "gamma": df_f * norm.pdf(d1) / (S * sigma * sq),
        "vega": S * df_f * norm.pdf(d1) * sq,
        "theta": -S * df_f * norm.pdf(d1) * sigma / (2 * sq) + rf * S * df_f * norm.cdf(d1) - rd * K * df_d * norm.cdf(d2),
    }

# Textbook reference (plan §3.8): S=K=1.60, rd=0.08, rf=0.11, sigma=0.20, T=0.5
print("reference call:", round(gk(1.60, 1.60, 0.08, 0.11, 0.20, 0.5)["call"], 6))

rng = np.random.default_rng(42)
cases = []
for _ in range(20):
    S = float(rng.uniform(0.5, 2.0))
    K = float(S * rng.uniform(0.8, 1.2))
    rd, rf = float(rng.uniform(-0.005, 0.06)), float(rng.uniform(-0.005, 0.06))
    sigma, T = float(rng.uniform(0.02, 0.4)), float(rng.uniform(0.05, 1.5))
    cases.append({"S": S, "K": K, "rd": rd, "rf": rf, "sigma": sigma, "T": T,
                  "expected": {k: float(v) for k, v in gk(S, K, rd, rf, sigma, T).items()}})
write_fixture("gk-crosscheck.json", {"source": "notebooks/crosscheck.ipynb (scipy.stats.norm)", "cases": cases})

reference call: 0.075053
wrote gk-crosscheck.json: 20 cases
